# Problem Set 01 — Pneumonia Detection from Chest X-Rays

Same pipeline as `src/`, run interactively. See `README.md` for the write-up.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow import keras
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Prepare the data

Strips `__MACOSX/` and `.DS_Store` (they break `image_dataset_from_directory`) and prints class counts.

In [ ]:
DATA_DIR = "../data/chest_xray"   # <- adjust
!python ../src/prepare_data.py --data-dir ../data

## 2. Build datasets

The shipped `val/` has 16 images, too few to select a model on, so we hold out 15% of `train/` instead.

In [ ]:
from dataset import make_datasets, compute_class_weights

IMG_SIZE, BATCH = (224, 224), 32
train_ds, val_ds, test_ds, orig_val_ds, class_names = make_datasets(DATA_DIR, IMG_SIZE, BATCH)
print("classes:", class_names, "-> positive =", class_names[1])

class_weights, counts = compute_class_weights(DATA_DIR, class_names)
print("train counts:", counts)
print("class weights:", class_weights)

## 3. Look at the images before modelling anything

In [ ]:
imgs, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, img, lab in zip(axes.ravel(), imgs, labels):
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(class_names[int(lab)]); ax.axis("off")
plt.tight_layout(); plt.show()

plt.bar(counts.keys(), counts.values(), color=["#4c78a8", "#f58518"])
plt.title("Training class distribution"); plt.ylabel("images"); plt.show()

## 4. Baseline CNN, trained from scratch

In [ ]:
from model import build_baseline, build_transfer, unfreeze_top

baseline = build_baseline(input_shape=IMG_SIZE + (3,))
baseline.summary()

In [ ]:
cbs = [
    keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7),
]
hist_base = baseline.fit(train_ds, validation_data=val_ds, epochs=25,
                         class_weight=class_weights, callbacks=cbs)

## 5. MobileNetV2 transfer learning

Stage 1 trains the head only; stage 2 unfreezes the top 40 backbone layers at lr=1e-5 with BatchNorm left frozen.

In [ ]:
transfer = build_transfer(input_shape=IMG_SIZE + (3,))
h1 = transfer.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weights, callbacks=cbs)

In [ ]:
transfer = unfreeze_top(transfer, n_layers=40, lr=1e-5)
h2 = transfer.fit(train_ds, validation_data=val_ds, epochs=23,
                  initial_epoch=len(h1.history["loss"]),
                  class_weight=class_weights, callbacks=cbs)

os.makedirs("../outputs/models", exist_ok=True)
transfer.save("../outputs/models/transfer.keras")

## 6. Evaluate

Thresholds are chosen on validation, then applied to test. Never the other way round.

In [ ]:
from evaluate import predict, pick_thresholds, report, plot_curves

y_val, p_val = predict(transfer, val_ds)
t_f1, t_rec = pick_thresholds(y_val, p_val, min_recall=0.95)
print(f"best-F1 threshold {t_f1:.3f} | recall>=0.95 threshold {t_rec:.3f}")

y_test, p_test = predict(transfer, test_ds)
for tag, thr in [("default", 0.5), ("best F1", t_f1), ("recall>=0.95", t_rec)]:
    report(y_test, p_test, thr, class_names, f"TEST @ {tag}")

In [ ]:
plot_curves(y_test, p_test, "../outputs/curves_roc_pr_notebook.png")
from IPython.display import Image; Image("../outputs/curves_roc_pr_notebook.png")

## 7. Grad-CAM

Confirms the model attends to lung fields rather than borders or annotations.

In [ ]:
from evaluate import save_gradcam
save_gradcam(transfer, test_ds, class_names, "../outputs/gradcam_notebook.png")
Image("../outputs/gradcam_notebook.png")

## 8. Findings

Record your numbers and observations here, then copy them into `README.md`.